<a href="https://colab.research.google.com/github/Sanskar-cpu-sudo/Deep-Learning/blob/main/Assignment-8/Assignemt_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERT for Sentiment Analysis

This notebook fine-tunes a pre-trained **BERT (bert-base-uncased)** model for binary sentiment classification using a sample of the **IMDB movie-review dataset**.

### Workflow
1. Install required libraries
2. Load a sample of the IMDB dataset
3. Tokenize text using the BERT tokenizer
4. Fine-tune pre-trained BERT for sentiment classification
5. Evaluate using Accuracy, Precision, Recall and F1-score
6. Test the trained model on new sentences


In [ ]:
!pip -q install -U transformers datasets accelerate scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 95.0 MB/s eta 0:00:00


In [ ]:
import random
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


## 1. Load the IMDB Dataset

The IMDB dataset contains movie reviews labelled as:
- **0 = Negative**
- **1 = Positive**

To make execution faster for classroom/lab use, we use a small sample. Increase the sample sizes if you have a GPU and want better performance.

In [ ]:
from datasets import load_dataset

dataset = load_dataset('stanfordnlp/imdb')

# Sample sizes: increase these for better accuracy on a GPU
TRAIN_SIZE = 2000
TEST_SIZE = 500

train_ds = dataset['train'].shuffle(seed=SEED).select(range(TRAIN_SIZE))
test_ds = dataset['test'].shuffle(seed=SEED).select(range(TEST_SIZE))

print('Training examples:', len(train_ds))
print('Testing examples :', len(test_ds))
print('\nExample review:\n', train_ds[0]['text'][:500])
print('Label:', train_ds[0]['label'])

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Training examples: 2000
Testing examples : 500

Example review:
 There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks Am
Label: 1


## 2. Load the Pre-trained BERT Tokenizer

In [ ]:
MODEL_NAME = 'bert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=256
    )

tokenized_train = train_ds.map(tokenize_function, batched=True)
tokenized_test = test_ds.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenizer('I really enjoyed this movie!'))


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

{'input_ids': [101, 1045, 2428, 5632, 2023, 3185, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


## 3. Load Pre-trained BERT for Classification

We use `AutoModelForSequenceClassification` with two output classes.

In [ ]:
id2label = {0: 'NEGATIVE', 1: 'POSITIVE'}
label2id = {'NEGATIVE': 0, 'POSITIVE': 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

print('Number of trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Number of trainable parameters: 109483778


## 4. Define Evaluation Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', zero_division=0
    )
    accuracy = accuracy_score(labels, predictions)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


## 5. Fine-tune BERT

The notebook uses a small batch size and one epoch so that it can run on a modest GPU. For stronger results, increase `num_train_epochs` to 2–3 and use more training examples.

In [ ]:
training_args = TrainingArguments(
    output_dir='./bert_sentiment_results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.296428,0.291211,0.872000,0.847328,0.902439,0.874016


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=250, training_loss=0.4403172836303711, metrics={'train_runtime': 39.6091, 'train_samples_per_second': 50.493, 'train_steps_per_second': 6.312, 'total_flos': 262736944328160.0, 'train_loss': 0.4403172836303711, 'epoch': 1.0})

## 6. Evaluate the Model

In [ ]:
metrics = trainer.evaluate()

print('Evaluation Results')
print('-' * 40)
for key, value in metrics.items():
    if isinstance(value, float):
        print(f'{key}: {value:.4f}')
    else:
        print(f'{key}: {value}')


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.296428,0.291211,1,0.872000,0.847328,0.902439,0.874016


Evaluation Results
----------------------------------------
eval_loss: 0.2912
eval_accuracy: 0.8720
eval_precision: 0.8473
eval_recall: 0.9024
eval_f1: 0.8740


In [ ]:
# Detailed classification report
pred_output = trainer.predict(tokenized_test)
predictions = np.argmax(pred_output.predictions, axis=-1)
true_labels = pred_output.label_ids

print(classification_report(
    true_labels,
    predictions,
    target_names=['NEGATIVE', 'POSITIVE'],
    digits=4
))


              precision    recall  f1-score   support

    NEGATIVE     0.8992    0.8425    0.8699       254
    POSITIVE     0.8473    0.9024    0.8740       246

    accuracy                         0.8720       500
   macro avg     0.8732    0.8725    0.8720       500
weighted avg     0.8737    0.8720    0.8719       500



## 7. Test on New Sentences


In [ ]:
def predict_sentiment(texts):
    model.eval()
    inputs = tokenizer(
        texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=256
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1)
        predictions = torch.argmax(probabilities, dim=-1)

    results = []
    for text, pred, probs in zip(texts, predictions, probabilities):
        results.append({
            'text': text,
            'sentiment': id2label[pred.item()],
            'confidence': float(probs[pred].item())
        })
    return results

sample_texts = [
    'This movie was absolutely fantastic. I loved every minute of it!',
    'The story was boring and the acting was terrible.',
    'It was an enjoyable movie with excellent performances.'
]

results = predict_sentiment(sample_texts)
for result in results:
    print(f"\nText      : {result['text']}")
    print(f"Sentiment : {result['sentiment']}")
    print(f"Confidence: {result['confidence']:.2%}")



Text      : This movie was absolutely fantastic. I loved every minute of it!
Sentiment : POSITIVE
Confidence: 94.78%

Text      : The story was boring and the acting was terrible.
Sentiment : NEGATIVE
Confidence: 90.54%

Text      : It was an enjoyable movie with excellent performances.
Sentiment : POSITIVE
Confidence: 94.99%


## 8. Save the Fine-tuned Model

The saved model can later be loaded without retraining.

In [ ]:
SAVE_PATH = './bert_sentiment_model'
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print('Model saved to:', SAVE_PATH)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./bert_sentiment_model


## Key Concepts

- **BERT**: A pre-trained Transformer language model that understands contextual relationships between words.
- **Fine-tuning**: Adapting the pre-trained BERT model to a specific task using labelled examples.
- **Tokenizer**: Converts text into BERT-compatible token IDs.
- **Sequence classification head**: Converts BERT representations into class predictions.
- **Softmax**: Converts output logits into class probabilities.
- **Accuracy / Precision / Recall / F1**: Metrics used to evaluate classification performance.
